## Day 5 

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits. 

Challenge: Translate the entire Brochure to German.

We will be provided a company name and their primary website.

In [1]:
import os
import json
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_contents, fetch_website_links
from openai import OpenAI

In [10]:
MODEL = 'nemotron-3-super:cloud'
openai = OpenAI(base_url='http://localhost:11434/v1/', api_key='ollama')

## First step: Have the model figure out which links are relevant

### Use a call to model to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [3]:
links = fetch_website_links("https://huggingface.co")
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/deepseek-ai/DeepSeek-V4-Pro',
 '/XiaomiMiMo/MiMo-V2.5-Pro',
 '/openai/privacy-filter',
 '/mistralai/Mistral-Medium-3.5-128B',
 '/talkie-lm/talkie-1930-13b-it',
 '/models',
 '/spaces/r3gm/wan2-2-fp8da-aoti-preview',
 '/spaces/smolagents/ml-intern',
 '/spaces/r3gm/wan2-2-fp8da-aoti-preview2',
 '/spaces/prithivMLmods/FireRed-Image-Edit-1.0-Fast',
 '/spaces/microsoft/TRELLIS.2',
 '/spaces',
 '/datasets/nvidia/Nemotron-Personas-Korea',
 '/datasets/Jackrong/GLM-5.1-Reasoning-1M-Cleaned',
 '/datasets/nvidia/Nemotron-Image-Training-v3',
 '/datasets/Roman1111111/claude-opus-4.6-10000x',
 '/datasets/open-thoughts/AgentTrove',
 '/datasets',
 '/join',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/inference/models',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/allenai',
 '/fa

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://huggingface.co"))


Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/login
/join
/spaces
/models
/deepseek-ai/DeepSeek-V4-Pro
/XiaomiMiMo/MiMo-V2.5-Pro
/openai/privacy-filter
/mistralai/Mistral-Medium-3.5-128B
/talkie-lm/talkie-1930-13b-it
/models
/spaces/r3gm/wan2-2-fp8da-aoti-preview
/spaces/smolagents/ml-intern
/spaces/r3gm/wan2-2-fp8da-aoti-preview2
/spaces/prithivMLmods/FireRed-Image-Edit-1.0-Fast
/spaces/microsoft/TRELLIS.2
/spaces
/datasets/nvidia/Nemotron-Personas-Korea
/datasets/Jackrong/GLM-5.1-Reasoning-1M-Cleaned
/datasets/nvidia/Nemotron-Image-Training-v3
/datasets/Roman1111111/claude-opus-4.6-10000x
/datasets/open-thoughts/AgentTrove
/datasets
/join
/enterprise
/enterprise
/enterprise

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling nemotron-3-super:cloud
Found 8 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co'},
  {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling nemotron-3-super:cloud
Found 7 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
7 days ago
•
535k
•
3.5k
XiaomiMiMo/MiMo-V2.5-Pro
Updated
6 days ago
•
11.8k
•
417
openai/privacy-filter
Updated
12 days ago
•
133k
•
1.24k
mistralai/Mistral-Medium-3.5-128B
Updated
2 days ago
•
12k
•
249
talkie-lm/talkie-1930-13b-it
Updated
11 days ago
•
216
Browse 2M+ models
Spaces
Running
on
Zero
MCP
2.52k
Wan2.2 14B Preview
🐌
2.52k
generate a video from an image with a text prompt
Running
on
CPU Upgrade
288
ML Intern
🤖
288
Chat with an AI‑powered ML Intern for quick help
Running
on

In [15]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling nemotron-3-super:cloud
Found 5 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n7 days ago\n•\n535k\n•\n3.5k\nXiaomiMiMo/MiMo-V2.5-Pro\nUpdated\n6 days ago\n•\n11.8k\n•\n417\nopenai/privacy-filter\nUpdated\n12 days ago\n•\n133k\n•\n1.24k\nmistralai/Mistral-Medium-3.5-128B\nUpdated\n2 days ago\n•\n12k\n•\n249\ntalkie-lm/talkie-1930-13b-it\nUpdated\n11 days ago\n•\n216\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nMCP\n2.52k\nWan

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="nemotron-3-super:cloud",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling nemotron-3-super:cloud
Found 8 relevant links


# Hugging Face: Where AI Gets a Hug (and a Hefty Dose of Fun)

## 🌟 The AI Community Building the Future  
*Yes, we really do love hugging faces—both the emoji kind and the transformer‑kind.*

- **2 M+ models** and growing faster than a meme on caffeine.  
- **500 k+ datasets** that let you train on everything from cat memes to climate data.  
- **Spaces** – your personal AI playground where you can spin up demos faster than you can say “zero‑shot”.  
- **Unlimited public hosting** for models, datasets, and apps—no gatekeepers, just pure collaborative joy.

---

## 🚀 Why Developers, Researchers, and Dreamers Choose Us  

| What You Get | Why It Matters |
|--------------|----------------|
| **Open‑source stack** | Build, tweak, and share without worrying about licensing fine print. |
| **Multimodal madness** | Text, image, video, audio, 3D—if it can be digitized, we’ve got a model for it. |
| **Instant portfolio** | Show off your work, garnish it with stars, and watch recruiters slide into your DMs. |
| **ML Intern (AI‑powered)** | Need a quick answer? Our friendly bot is basically the overeager intern who never sleeps. |
| **Community vibe** | Hackathons, forums, and meme‑filled discussions make learning feel like a party. |

---

## 🏢 Enterprise & Team Plans – AI at Scale, Without the Corporate Snooze  

- **Team Plan** – starts at **$20/user/month** (because even AI needs a budget-friendly coffee break).  
- **Enterprise Plan** – talk to sales for flexible contracts that hug your security, compliance, and scalability needs.  
- **SSO Integration** – log in once, rule them all (your IdP, not the One Ring).  
- **Regions & Audit Logs** – keep your data where you want it, and know who did what, when, and why (no mystery AI ghosts).  
- **Dedicated Support** – because even the best models occasionally need a human hug.

---

## 👩‍💻 Life at Hugging Face – Culture, Careers, and Inside Jokes  

- **Open‑source first, ego second** – we celebrate contributions, not corner offices.  
- **Quirky perks** – think weekly “model show‑and‑tell”, meme Fridays, and a Slack channel devoted to pet‑AI pics.  
- **Career opportunities** – from research engineers to developer advocates, we’re hiring humans who love to teach machines (and occasionally teach each other a new dance move).  
- **Learning on the job** – access to cutting‑edge models, datasets, and compute means you’ll level up faster than a transformer in a GPU farm.  
- **Global vibe** – offices (and remote hubs) spread across continents, so you can collaborate while sipping espresso in Paris or matcha in Tokyo.

---

## 📣 Join the Hug  

Whether you’re a solo tinkerer, a startup looking to ship AI fast, or a Fortune‑500 enterprise needing iron‑clad security, Hugging Face has a place for you—complete with emojis, open‑source love, and a community that treats every pull request like a high‑five.

**Ready to get hugged?**  
[Sign Up]() • [Explore Models]() • [Check out Enterprise]()  

*Let’s build the future—one hug (and one model) at a time.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling nemotron-3-super:cloud
Found 3 relevant links


# 🤗 Hugging Face – Where AI Gets a Hug (and a High‑Five)

*Welcome to the place where models frolic, datasets dance, and Spaces spin like a disco ball. If you think AI is all serious math and no memes, think again.*

---

## 🎉 What We Are

- **The AI community building the future** – 2 M+ models, 500 k+ datasets, and a growing universe of Spaces waiting for your next wild idea.  
- **Open‑source playground** – HF Transformers, Tokenizers, Accelerate, and the whole “🤗” stack are free to fork, tweak, and brag about on LinkedIn.  
- **Enterprise‑grade, but still friendly** – SSO, region‑pinned data, audit logs, and dedicated support—so your CISO can sleep while your data scientists party.

> *“We treat every pull request like a birthday cake: sweet, shared, and best enjoyed with sprinkles (of code).”*

---

## 👥 Who’s Hanging Out Here?

| Crowd | Why They Love Us |
|-------|------------------|
| **Researchers & Academics** | State‑of‑the‑art models at a click; reproducible papers without the paper‑jam. |
| **Startup Hackers** | Spin up a demo Space in minutes, show investors a working AI app before lunch. |
| **Enterprises** | Scale model serving with Team ($20/user/mo) or custom Enterprise plans—security that actually *gets* you. |
| **Students & Hobbyists** | Free compute, tutorials, and a community that will answer your “why does my loss look like a Picasso?” questions at 2 a.m. |
| **AI‑Curious Cats** | Because even cats need a place to showcase their GPT‑powered mouse‑generator. |

---

## 💼 Life at Hugging Face (Culture, Perks & Inside Jokes)

- **Open‑Source First, Meetings Second** – We believe the best ideas live in a repo, not a slide deck.  
- **Weekly “Model‑Mash” Fridays** – Show off the weirdest fine‑tune you’ve cooked up; prizes include stickers, bragging rights, and occasional pizza.  
- **Remote‑First, Yet We Still Hug** – Virtual coffee chats, async stand‑ups, and the occasional actual hug when you visit our Paris/NYC/SF hubs.  
- **Learning Stipend** – Buy books, courses, or that GPU you’ve been eyeing—because we know you’ll spend it on more models anyway.  
- **Diversity & Inclusion** – We celebrate every pronoun, every background, and every weird model architecture (looking at you, Mixture‑of‑Experts).  
- **Pet‑Friendly Slack** – #random is filled with cat‑GANs, dog‑detectors, and the occasional office iguana that insists on reviewing pull requests.

> *“If you can’t explain your model to a rubber duck, you haven’t tried hard enough… or you haven’t met our office duck yet.”*

---

## 🚀 Careers – Join the 𝙷𝚞𝚐𝚐𝚒𝚗𝚐 🙂 Crew

We’re always hunting for bright minds who love code, community, and a good meme. Current openings (check the **Careers** page for the latest):

- **ML Engineer / Research Scientist** – Push the frontier of transformers, diffusion, or whatever the next hot thing is.  
- **Developer Relations / Community Engineer** – Turn strangers into contributors, one tutorial at a time.  
- **Site Reliability Engineer** – Keep our Spaces humming and our buckets from spilling over.  
- **Product Manager (Enterprise)** – Shape the features that make Fortune‑500 CTOs whisper, “Damn, that’s slick.”  
- **UX/UI Designer** – Make model cards look as gorgeous as the outputs they describe.  
- **People Ops / Culture Champion** – Keep the hugs coming and the vibes high.

*Perks:* competitive salary, equity, health‑wellness stipend, unlimited PTO (yes, really), and a lifetime supply of virtual high‑fives.

---

## 📈 Why Investors & Partners Should Pay Attention

- **Network Effect:** 2 M+ models → more contributors → better models → more users. Virtuous cycle, powered by git.  
- **Revenue Streams:** Team subscription, Enterprise contracts, and premium compute (GPU/TPU on demand).  
- **Strategic Partnerships:** Tight integrations with cloud providers, hardware vendors, and the occasional AI‑startup incubator.  
- **Brand Love:** The 🤗 emoji is now a universal sign for “I’m about to show you something cool.”  

---

## 📖 TL;DR – The Hugging Face Pitch in One Sentence

> *We’re the open‑source, community‑driven, slightly weird, and seriously powerful platform where anyone can host, share, and serve AI—complete with enterprise‑grade security, a culture that celebrates both code and silliness, and enough models to keep your GPU busy until the heat death of the universe.*

---

**Ready to dive in?**  
👉 **Create an account**, upload your first model, spin up a Space, and let the world give your AI a hug.  

*See you on the hub!* 🚀🤗

## Translate the brochure to German

In [24]:
def translate_brochure_to_german(brochure_text):
    translation_system_prompt = "You are a professional translator. Translate the following company brochure to German, maintaining all formatting and structure. Provide only the translated brochure without any explanations."
    
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": translation_system_prompt},
            {"role": "user", "content": f"Please translate this brochure to German:\n\n{brochure_text}"}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
brochure_response = openai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": brochure_system_prompt},
        {"role": "user", "content": get_brochure_user_prompt("HuggingFace", "https://huggingface.co")}
    ],
)
brochure_text = brochure_response.choices[0].message.content

print("🇩🇪 Translating to German...\n")
translate_brochure_to_german(brochure_text)

Selecting relevant links for https://huggingface.co by calling nemotron-3-super:cloud
Found 4 relevant links
🇩🇪 Translating to German...



# 🤗 Hugging Face  
### *Wo KI eine Umarmung und ein High‑Five bekommt*  

---

## 🌐 Wer Wir Sind  
Hugging Face ist das **Kollaborations‑Hub** für die Machine‑Learning‑Community – ein lebhafter Basar aus Modellen, Datensätzen und Spaces, in dem jeder teilen, experimentieren und seine KI‑Zauberkünste zeigen kann. Stellt euch uns als das **GitHub der KI** vor, nur mit mehr Emojis und weniger Merge‑Konflikten.

- **Mission:** Gemeinsam eine offene, ethische KI‑Zukunft aufbauen.  
- **Vibe:** Open‑Source‑Herz, Enterprise‑Kraft und eine Community, die jedes PR wie einen Geburtstagskuchen behandelt.  

---

## 🚀 Was Wir Anbieten  

| Feature | Warum Es Großartig Ist |
|---------|------------------------|
| **2M+ Modelle** | Von winzigen Transformern bis zu riesigen LLMs – wählt euer Gift, wir haben den Geschmack. |
| **500k+ Datensätze** | Braucht ihr koreane Personas, Bild‑Trainingsdaten oder einen Datensatz voller Katzen‑Memes? Wir haben ihn. |
| **Spaces** | Wandelt ein Notebook schneller in eine Live‑Demo um, als ihr „Zero‑GPU“ sagen könnt. |
| **Enterprise & Team Pläne** | Sicheres SSO, Audit‑Logs, privater Speicher und genug ZeroGPU‑Quota, um eure Data Scientists zum Tanz zu bringen. |
| **HF Open‑Source‑Stack** | Bibliotheken, Inferenz‑Endpunkte und HuggingChat – alles frei nutzbar, modificierbar und zum Schreien in die Leere. |

---

## 👥 Wer Uns Nutzt  

- **Start‑ups**, die die nächste virale KI‑App in einer Garage (oder einem Co‑Working‑Space mit zweifelhaftem Kaffee) prototypen.  
- **Fortune‑500**‑Konzerne, die KI verantwortungsvoll skalieren – denn sogar der größte Fisch benötigt einen schönen Open‑Source‑Teich.  
- **Forschende**, die Durchbrüche veröffentlichen, während sie gleichzeitig ihr öffentliches ML‑Profil aufbauen (hallo, LinkedIn‑Prahlrechte).  
- **Bildende**, die Theorie in praxisnahe Labs verwandeln, ohne einen Doktortitel in DevOps zu benötigen.  

Kurz gesagt: *Wer Daten atmet, gehört bereits zur Familie.*

---

## 💼 Leben bei Hugging Face  

- **Kultur:** Neugierig, kollaborativ und leicht meme‑besessen. Wir sehen Misserfolge als Lernchancen (und als zusätzliches Material für unseren internen Meme‑Channel).  
- **Vorteile:** Flexible Remote‑Arbeit, unbegrenztes Lernbudget, quartalsweise „Hug‑Tage“, bei denen wir Code deployen und anschließend tatsächlich einander umarmen (virtuell, selbstverständlich).  
- **Diversität:** Wir suchen aktiv Stimmen aus allen Hintergründen – denn die beste KI entsteht aus einem Chor, nicht einem Solisten.  
- **Wachstum:** Klare Karrierestufen, Mentoring‑Programme und die Möglichkeit, an Projekten zu arbeiten, die wirklich *zählen* (keine weiteren „einfach nur CRUD‑Apps“, außer ihr wollt das wirklich).  

**Offene Positionen** (Stand aktueller Infos):  
- ML Engineer (Model Hub)  
- Developer Advocate (Spaces & Community)  
- Security Engineer (Enterprise Trust)  
- Product Designer (UX für KI)  
- …und viele mehr – schaut auf unserer Karriereseite für die vollständige Liste vorbei.

---

## 📈 Warum Investieren?  

- **Netzwerkeffekte:** Mehr Modelle → mehr Nutzer → mehr Modelle. Ein tugendhafter Kreis, der unsere Plattform klebrig macht.  
- **Umsatzströme:** Team‑ & Enterprise‑Abonnements, Compute‑Credits und Premium‑Services – alles skaliert mit dem KI‑Boom.  
- **Talentmagnet:** Die Besten und Hellsten wollen dort arbeiten, wo sie Code shippen können, der tatsächlich von Millionen genutzt wird.  
- **Zukunftssicher:** Wir sind modality‑agnostisch (Text, Bild, Video, Audio, 3D). Egal, welche nächste KI‑Welle kommt, wir surfen bereits darauf.  

---

## 📣 Mach Mit beim Hug  

Egal, ob ihr **bauen**, **investieren** oder einfach nur über den neuesten Modell‑Release **geeken** wollt – Hugging Face heißt euch mit offenen Armen (und einer virtuellen Umarmung) willkommen.  

👉 **Anmelden** unter [huggingface.co](https://huggingface.co)  
👉 **Entdecken** Modelle, Datensätze, Spaces  
👉 **Schaut vorbei** auf unserer [Karriere](https://huggingface.co/careers) Seite für euer nächstes Abenteuer  

*Lasst uns KI freundlich, offen und ein bisschen lächerlich machen – zusammen.* 🚀🤗